# Set up the fire danger site for a new area

This notebook turns your copy of this repository into the FDRA fire danger site for **your** area: the map of FDRAs, the daily popups (fire danger, staffing, adjective and dispatch levels), the season-tracking charts and the FDOP Workbench. It also sets up the thresholds workbook your planners edit afterwards.

**Before you start**
1. Make your own copy of the repository: on GitHub open the template repository, click **Use this template → Create a new repository**, and give it a name (for example `NOCO-fire-danger`). Make it **public** (GitHub Pages and the raw data links need it).
2. Open your new repository in **GitHub Codespaces** (green **Code** button → **Codespaces** → **Create codespace on main**). Nothing to install on your computer. You can also run this notebook in your own Jupyter after `git clone`.
3. In the codespace, open `setup/setup_area.ipynb`, choose the **Python 3** kernel when asked, and run the cells top to bottom (Shift+Enter).

**What you need**
- FDRA boundaries: shapefiles (zipped is fine), GeoJSON or a file geodatabase. Put them in `setup/inputs/` (drag them into the file panel).
- Each FDRA's SIG stations (WIMS / NWS ids) and fuel model.
- The FDOP thresholds, if you want them loaded now. You can also start with none: the site then uses 90th / 97th percentiles until thresholds are added to the workbook.

Every step writes ordinary files in the repository, so you can re-run a step or fix a file by hand. Nothing is published until the last step pushes the changes.

The worked example is the **Northern Colorado (NoCo) 2026 FDOP**. Leave `EXAMPLE = 'noco'` to see every step with real inputs, or set `EXAMPLE = None` and fill in your own.

In [ ]:
%pip install -q geopandas pyogrio shapely openpyxl requests pandas folium mapclassify

In [ ]:
import sys, subprocess, importlib
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'docs' / 'assets' / 'area.js').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / 'setup'), str(ROOT / 'site'), str(ROOT / 'setup' / 'examples' / 'noco')]
import area_setup as S; importlib.reload(S)
import pandas as pd
pd.set_option('display.max_rows', 200)

EXAMPLE = 'noco'          # set to None for your own area
if EXAMPLE == 'noco':
    import noco_fdop as X; importlib.reload(X)
print('Repository:', ROOT)

## Step 1 · Area settings

- `id`: a short code shown on the pages (e.g. `NOCO`).
- `repo`: your GitHub repository as `owner/name` (read from the codespace if left blank).
- `states`: two-letter states the FDRAs are in. They select the RAWS stations and the fire history.
- `timezone`: an IANA name such as `America/Denver`.
- The map extent is worked out from the FDRAs.

In [ ]:
def git_repo():
    try:
        url = subprocess.run(['git', 'remote', 'get-url', 'origin'], cwd=ROOT, capture_output=True, text=True).stdout.strip()
        return url.rstrip('/').removesuffix('.git').split('github.com')[-1].lstrip('/:')
    except Exception:
        return ''

if EXAMPLE == 'noco':
    AREA = dict(X.AREA)
else:
    AREA = {
        'id': 'XX', 'name': 'My Area',
        'repo': '',                       # owner/name — blank = this codespace's repository
        'states': ['CO'],
        'timezone': 'America/Denver',
    }
AREA['repo'] = git_repo() or AREA.get('repo')
print(AREA)

## Step 2 · FDRA boundaries

Put the boundary files in `setup/inputs/`. Then set:
- `SHAPES`: the files to read (one file with all FDRAs, or one per FDRA).
- `NAME_FIELD`: the column that names the FDRA. Shapes with the same name are merged, for example response zones into their FDRA.

The cell below lists the columns and values it finds, to help you choose.

In [ ]:
if EXAMPLE == 'noco':
    SHAPES = [ROOT / 'setup' / 'examples' / 'noco' / X.SHAPES]
    NAME_FIELD = X.NAME_FIELD
else:
    SHAPES = sorted((ROOT / 'setup' / 'inputs').glob('*.zip')) + sorted((ROOT / 'setup' / 'inputs').glob('*.shp')) + \
             sorted((ROOT / 'setup' / 'inputs').glob('*.geojson'))
    NAME_FIELD = 'FDRA'
shapes = S.read_shapes(SHAPES)
print(len(shapes), 'shapes from', [p.name for p in SHAPES])
print('columns:', [c for c in shapes.columns if c != 'geometry'])
print(NAME_FIELD, 'values:', sorted(shapes[NAME_FIELD].astype(str).unique()) if NAME_FIELD in shapes else 'NOT FOUND — set NAME_FIELD')

Each FDRA gets an id `PREFIX-CODE` (for example `NOCO-JU`). The id is used in links and in the thresholds workbook, so keep it short and don't change it later. Set `NAMES` to choose each FDRA's code and display name.

In [ ]:
if EXAMPLE == 'noco':
    PREFIX, NAMES, FOREST = X.PREFIX, X.NAMES, X.FOREST
else:
    PREFIX = AREA['id']
    NAMES = {}            # {'Front Range': ('FR', 'Front Range'), ...}; blank = codes from the initials
    FOREST = AREA['name']
fdras = S.build_fdras(shapes, NAME_FIELD, PREFIX, names=NAMES, forest=FOREST, crs=AREA.get('crs', S.ALBERS))
ov = S.overlaps(fdras, AREA.get('crs', S.ALBERS))
print('Overlapping FDRAs (should be none):', ov or 'none')
fdras.drop(columns='geometry')[['fdra_id', 'fdra_name', 'forest', 'acres']]

In [ ]:
try:
    display(fdras.explore(column='fdra_id', tooltip=['fdra_id', 'fdra_name', 'acres'], style_kwds={'weight': 2}))
except Exception as e:
    print('Map preview needs folium:', e)

## Step 3 · SIG stations

The national public RAWS layer gives every station in your states. It shows each station's WIMS id, whether it is active, and when it last reported. The first table suggests the stations inside each FDRA.

Then list each FDRA's SIG stations (`fdra_id, stationId, name, fuel_model`). The check looks up each id on the RAWS list and asks FEMS for a week of data. When an id looks wrong, it suggests the id the RAWS list has for a station of the same name. Fix the list and run the cell again until every row says `ok`.

In [ ]:
raws = S.raws_for_states(AREA['states'])
print(len(raws), 'RAWS in', AREA['states'], '·', (raws.Status == 'A').sum(), 'active')
S.stations_inside(fdras, raws).sort_values(['fdra_id', 'km_outside', 'name'])

In [ ]:
if EXAMPLE == 'noco':
    members = pd.DataFrame(X.STATIONS, columns=['fdra_id', 'stationId', 'name', 'fuel_model'])
else:
    f = ROOT / 'setup' / 'inputs' / 'stations.csv'
    if not f.exists():
        pd.DataFrame(columns=['fdra_id', 'stationId', 'name', 'fuel_model']).to_csv(f, index=False)
        print('Fill in', f, 'and run this cell again')
    members = pd.read_csv(f)
unknown = sorted(set(members.fdra_id) - set(fdras.fdra_id))
print('FDRA ids in the station list that are not FDRAs:', unknown or 'none')
chk = S.check_stations(members, raws)
print('Stations OK:', int(chk.ok.sum()), 'of', len(chk))
chk

## Step 4 · Save the FDRAs, stations and area settings

In [ ]:
S.write_membership(members, fdras)
S.write_fdras(fdras)
area = S.write_area(AREA)
print('Wrote', S.FDRAS.relative_to(ROOT), S.MEMBERSHIP.relative_to(ROOT), 'and docs/assets/area.js')
print('Site will be at', area['site_url'])

## Step 5 · Thresholds workbook

This writes `site/thresholds/FDRA_thresholds.xlsx` with a row for every FDRA. It is the one place to change thresholds afterwards: download it from GitHub, edit it in Excel, and upload it again. Each upload is checked automatically. The **How to update** sheet inside explains every column.

The workbook handles these kinds of tool:
- **Single-index breakpoints** (ERC, BI, IC, SC, KBDI, GSI, 100-hr / 1000-hr FM), on the **Thresholds** sheet.
- **Matrices**, such as ERC rows × BI columns, on the **Matrix tools** sheet.
- **Multi-day averages** (Days used, e.g. `O1+F1+F2+F3`) and **one-class adjustments** (e.g. `Tmax < 57 : -1`), on either sheet.

With `EXAMPLE = 'noco'` the workbook is filled from the NoCo 2026 FDOP. Otherwise it starts with only the FDRA rows, and you add the thresholds in Excel.

In [ ]:
S.clear_template_content(AREA['name'])
start = X.start() if EXAMPLE == 'noco' else {}
book, problems = S.new_workbook(start)
d, problems = S.check_workbook()
print('FDRAs with thresholds:', len(d))

## Step 6 · Publish

This commits the files and pushes them to GitHub. The push starts the builds:
- **Site data** (popups, charts, today's levels): the first run also builds the 2016–2025 normals, about 20–40 minutes.
- **FDOP data** (the Workbench's station history and fire occurrence): about 30–60 minutes.
- **Thresholds check**.

Then turn on the website, once: in the repository go to **Settings → Pages**, set **Source: Deploy from a branch**, **Branch: main**, **Folder: /docs**, and click Save.

In [ ]:
def git(*a):
    r = subprocess.run(['git', *a], cwd=ROOT, capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    return r.returncode
git('add', '-A', 'docs', 'site', 'setup', 'agol')
git('commit', '-m', f"Set up {AREA['name']} ({AREA['id']}) with setup_area.ipynb")
git('push')
print('Website (after Pages is on and the first build finishes):', area['site_url'])

Optional: with the GitHub CLI (already signed in inside a codespace) you can turn on Pages and start the builds from here instead.

In [ ]:
# !gh api -X POST repos/{AREA['repo']}/pages -f 'source[branch]=main' -f 'source[path]=/docs'
# !gh workflow run "Site data" -R {AREA['repo']} -f rebuild_normals=true
# !gh workflow run "FDOP data" -R {AREA['repo']}

## Step 7 · Optional: ArcGIS Online layer and web map

This publishes the FDRAs as a hosted feature layer in your ArcGIS Online organization, and makes a web map from it. Clicking an FDRA in the map shows today's fire danger, staffing, adjective and dispatch levels from the site build, with links to the charts and the Workbench. You sign in with your own account (you are asked for your password; it is not saved anywhere). Build an Experience Builder app on the web map if you want one.

In [ ]:
# %pip install -q arcgis
# from arcgis.gis import GIS
# gis = GIS('https://nifc.maps.arcgis.com', input('ArcGIS Online username: '))
# layer, webmap = S.agol_publish(gis, area, group=None)     # group = a group id to share with
# print(layer.homepage, webmap.homepage)

## After setup

- **Update thresholds:** `site/thresholds/README.md` (download the workbook, edit, upload; checked on upload).
- **Change FDRA boundaries or stations:** run steps 2–4 again and push.
- **Fire history** comes from the FPA FOD and WFIGS for your states, and burned area from the NIFC interagency perimeter history.